# OSNet-AIN x1.0 — вариант 4: обучающая стратегия

Ноутбук последовательно проверяет MixStyle, визуально сложные негативы, динамический вес metric loss и Circle Loss. Screening использует только внутренний split внутри train; outer calibration/validation остаются скрыты до финального сравнения по трём seed. Test-набор не используется.

## 1. Настройка

Ячейки можно выполнить через `Run All`. Завершённые подэтапы будут прочитаны из JSON, незавершённые продолжатся из `last.pt`.

In [1]:
import gc
import json
import os
import sys
from dataclasses import replace
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'backend').exists():
    ROOT = ROOT.parent
assert (ROOT / 'backend').exists(), 'Не найден корень Car-classification-MSK'
os.chdir(ROOT)

import torch
from backend.core import DATASET
from training.hpo import experiment_losses, make_optimizer
from training.pipeline import ensure_splits, select_device, split_summary
from training.stage4 import (
    ACTIVE_REFERENCE, SimilarityPKBatchSampler, base_stage4_config,
    export_stage4_winner, initialize_from_base_checkpoint, prepare_strategy,
    run_final_comparison,
    run_strategy_screening,
)

VARIANT_DIR = ROOT / 'OSNet-AIN-x1.0/variant_04_training_strategy'
RESULTS_DIR = VARIANT_DIR / 'results'
WEIGHTS_DIR = VARIANT_DIR / 'weights'
BASE_CHECKPOINT = (ROOT / 'OSNet-AIN-x1.0/variant_02_hpo_bnneck_supcon/'
                   'weights/selected_run_02/best_map.pt')
SEEDS = (20260915, 20260916, 20260917)
SCREEN_EPOCHS = 8
FINAL_EPOCHS = None  # взять best_epoch победителя inner screening
RUN_SCREENING = True
RUN_FINAL_COMPARISON = True
RUN_EXPORT = True
DEVICE = select_device()

assert BASE_CHECKPOINT.exists(), f'Не найден {BASE_CHECKPOINT}'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
print('python:', sys.executable)
print('device:', DEVICE, '| torch:', torch.__version__)
print('screening:', SCREEN_EPOCHS, 'эпох | final: inner-selected | seeds:', SEEDS)

python: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python
device: mps | torch: 2.14.0
screening: 8 эпох | final: inner-selected | seeds: (20260915, 20260916, 20260917)


## 2. Данные и контроль протокола

Здесь должны появиться те же 925 train identity, 307 calibration и 309 validation.

In [2]:
rows, split = ensure_splits(DATASET)
print(json.dumps(split_summary(rows, split), ensure_ascii=False, indent=2))
assert len(split['identities']['train']) == 925
assert set(split['identities']['train']).isdisjoint(split['identities']['calibration'])
assert set(split['identities']['train']).isdisjoint(split['identities']['validation'])

{
  "train": {
    "identities": 925,
    "images": 5717
  },
  "calibration": {
    "identities": 307,
    "images": 1903
  },
  "validation": {
    "identities": 309,
    "images": 1936
  }
}


## 3. Smoke test

Один реальный optimizer step проверяет MixStyle и Circle Loss на выбранном устройстве. Отдельная синтетическая проверка подтверждает форму P×K hard-negative batch.

In [3]:
smoke_config = replace(
    base_stage4_config(BASE_CHECKPOINT, epochs=1, seed=SEEDS[0]),
    use_mixstyle=True, metric_loss='circle', loss_weight_schedule='metric_warmup')
smoke_rows, smoke_labels, smoke_sampler, smoke_loader = prepare_strategy(
    rows, split['identities']['train'], smoke_config, dataset=DATASET)
smoke_model, loaded = initialize_from_base_checkpoint(
    BASE_CHECKPOINT, len(smoke_labels), smoke_config, DEVICE,
    load_classifier=True)
clean, robust, labels, _ = next(iter(smoke_loader))
optimizer = make_optimizer(smoke_model, smoke_config)
optimizer.zero_grad(set_to_none=True)
losses = experiment_losses(
    smoke_model, clean.to(DEVICE), robust.to(DEVICE), labels.to(DEVICE), smoke_config)
losses['loss'].backward()
optimizer.step()
assert torch.isfinite(losses['loss'])
assert loaded > 500

synthetic = [{'label': label, 'camera_id': camera}
             for label in range(4) for camera in (1, 2)]
neighbors = {0: [1, 2, 3], 1: [0, 2, 3],
             2: [3, 0, 1], 3: [2, 0, 1]}
hard_sampler = SimilarityPKBatchSampler(
    synthetic, identities_per_batch=2, images_per_identity=2,
    neighbors=neighbors, seed=7)
assert all(len(batch) == 4 for batch in hard_sampler)
print('smoke loss:', float(losses['loss'].detach()), '| checkpoint tensors:', loaded)

del smoke_model, smoke_loader, smoke_sampler, optimizer, clean, robust, labels, losses
gc.collect()
if DEVICE.type == 'mps': torch.mps.empty_cache()
if DEVICE.type == 'cuda': torch.cuda.empty_cache()

smoke loss: 56.97407531738281 | checkpoint tensors: 565


## 4. Screening на внутреннем train split

Порядок: baseline → MixStyle → hard negatives → при необходимости их комбинация → dynamic weights → Circle Loss. Выбор делается только по raw mAP@10 внутренней validation-части.

In [4]:
screening_path = RESULTS_DIR / 'screening_summary.json'
if RUN_SCREENING:
    screening = run_strategy_screening(
        rows, split, DEVICE, BASE_CHECKPOINT, RESULTS_DIR, WEIGHTS_DIR,
        epochs=SCREEN_EPOCHS, dataset=DATASET)
else:
    screening = json.loads(screening_path.read_text())

assert screening['outer_calibration_or_validation_used'] is False
print('Победитель screening:', screening['winner'])
for name, result in screening['candidates'].items():
    print(f"{name:28s} best inner mAP={result['best_mAP']:.6f} "
          f"epoch={result['best_epoch']}")

epoch 1:   0%|          | 0/46 [00:00<?, ?it/s]

Epoch 1/8 | осталось эпох: 7 | эпоха: 00:00:44 | прошло: 00:00:44 | ETA: 00:05:09 | best mAP: 0.9484


epoch 2:   0%|          | 0/46 [00:00<?, ?it/s]

Epoch 2/8 | осталось эпох: 6 | эпоха: 00:00:45 | прошло: 00:01:29 | ETA: 00:04:27 | best mAP: 0.9484


epoch 3:   0%|          | 0/46 [00:00<?, ?it/s]

...

Epoch 6/8 | осталось эпох: 2 | эпоха: 00:00:44 | прошло: 00:04:28 | ETA: 00:01:29 | best mAP: 0.9426


epoch 7:   0%|          | 0/46 [00:00<?, ?it/s]

Epoch 7/8 | осталось эпох: 1 | эпоха: 00:00:44 | прошло: 00:05:12 | ETA: 00:00:45 | best mAP: 0.9426


epoch 8:   0%|          | 0/46 [00:00<?, ?it/s]

Epoch 8/8 | осталось эпох: 0 | эпоха: 00:00:45 | прошло: 00:05:57 | ETA: 00:00:00 | best mAP: 0.9426
Победитель screening: baseline
baseline                     best inner mAP=0.948353 epoch=1
mixstyle                     best inner mAP=0.941944 epoch=1
hard_negatives               best inner mAP=0.947580 epoch=1
dynamic_weights              best inner mAP=0.944568 epoch=1
circle_loss                  best inner mAP=0.942552 epoch=1


## 5. Финальное matched-сравнение по трём seed

Контроль и выбранная стратегия имеют одинаковые базовые гиперпараметры. Для каждого seed реранкинг и порог подбираются на calibration, после чего конфигурация один раз оценивается на validation.

In [5]:
comparison_path = RESULTS_DIR / 'final_comparison.json'
if RUN_FINAL_COMPARISON:
    comparison = run_final_comparison(
        rows, split, screening, DEVICE, BASE_CHECKPOINT, RESULTS_DIR, WEIGHTS_DIR,
        seeds=SEEDS, epochs=FINAL_EPOCHS, dataset=DATASET)
else:
    comparison = json.loads(comparison_path.read_text())

print('Победитель matched-сравнения:', comparison['winner'])
for name, result in comparison['aggregates'].items():
    print(name,
          'mean mAP@10=', round(result['mean_mAP_at_10'], 6),
          'std=', round(result['std_mAP_at_10'], 6),
          'candidate=', round(result['mean_candidate_score'], 6),
          'quality=', round(result['mean_quality_score'], 6))
print('Текущий активный MVP:', json.dumps(ACTIVE_REFERENCE, ensure_ascii=False))
print('Превзойдён активный MVP:', comparison['beats_active_reference'])
if not comparison['beats_active_reference']:
    print('Рекомендация: не заменять активные веса MVP результатом этапа 4.')

epoch 1:   0%|          | 0/57 [00:00<?, ?it/s]

Epoch 1/1 | осталось эпох: 0 | эпоха: 00:00:44 | прошло: 00:00:44 | ETA: 00:00:00
Calibration reranking: k1=10, k2=1
Calibration reranking: k1=10, k2=3
Calibration reranking: k1=10, k2=6
Calibration reranking: k1=20, k2=1
Calibration reranking: k1=20, k2=3
Calibration reranking: k1=20, k2=6
Calibration reranking: k1=30, k2=1
Calibration reranking: k1=30, k2=3
Calibration reranking: k1=30, k2=6


epoch 1:   0%|          | 0/57 [00:00<?, ?it/s]

Epoch 1/1 | осталось эпох: 0 | эпоха: 00:00:43 | прошло: 00:00:43 | ETA: 00:00:00
Calibration reranking: k1=10, k2=1
Calibration reranking: k1=10, k2=3
Calibration reranking: k1=10, k2=6
Calibration reranking: k1=20, k2=1
Calibration reranking: k1=20, k2=3
Calibration reranking: k1=20, k2=6
Calibration reranking: k1=30, k2=1
Calibration reranking: k1=30, k2=3
Calibration reranking: k1=30, k2=6


epoch 1:   0%|          | 0/57 [00:00<?, ?it/s]

...

Epoch 1/1 | осталось эпох: 0 | эпоха: 00:00:43 | прошло: 00:00:43 | ETA: 00:00:00
Calibration reranking: k1=10, k2=1
Calibration reranking: k1=10, k2=3
Calibration reranking: k1=10, k2=6
Calibration reranking: k1=20, k2=1
Calibration reranking: k1=20, k2=3
Calibration reranking: k1=20, k2=6
Calibration reranking: k1=30, k2=1
Calibration reranking: k1=30, k2=3
Calibration reranking: k1=30, k2=6


epoch 1:   0%|          | 0/57 [00:00<?, ?it/s]

Epoch 1/1 | осталось эпох: 0 | эпоха: 00:00:44 | прошло: 00:00:44 | ETA: 00:00:00
Calibration reranking: k1=10, k2=1
Calibration reranking: k1=10, k2=3
Calibration reranking: k1=10, k2=6
Calibration reranking: k1=20, k2=1
Calibration reranking: k1=20, k2=3
Calibration reranking: k1=20, k2=6
Calibration reranking: k1=30, k2=1
Calibration reranking: k1=30, k2=3
Calibration reranking: k1=30, k2=6


epoch 1:   0%|          | 0/57 [00:00<?, ?it/s]

Epoch 1/1 | осталось эпох: 0 | эпоха: 00:00:44 | прошло: 00:00:44 | ETA: 00:00:00
Calibration reranking: k1=10, k2=1
Calibration reranking: k1=10, k2=3
Calibration reranking: k1=10, k2=6
Calibration reranking: k1=20, k2=1
Calibration reranking: k1=20, k2=3
Calibration reranking: k1=20, k2=6
Calibration reranking: k1=30, k2=1
Calibration reranking: k1=30, k2=3
Calibration reranking: k1=30, k2=6
Победитель matched-сравнения: control
control mean mAP@10= 0.796863 std= 0.007137 candidate= 0.752816 quality= 0.43387
strategy mean mAP@10= 0.796863 std= 0.007137 candidate= 0.752816 quality= 0.43387
Текущий активный MVP: {"mAP_at_10": 0.8146886982413298, "candidate_score": 0.747121224071299, "quality_score": 0.44132203661572833}
Превзойдён активный MVP: False
Рекомендация: не заменять активные веса MVP результатом этапа 4.


## 6. Экспериментальный ONNX-экспорт

Экспорт сохраняется только внутри варианта 4 и не меняет `models/` или backend. Даже при отрицательном результате он полезен для воспроизводимости эксперимента.

In [6]:
if RUN_EXPORT:
    export_summary = export_stage4_winner(
        comparison, split, rows, WEIGHTS_DIR,
        WEIGHTS_DIR / 'osnet_stage4_selected.onnx',
        RESULTS_DIR / 'export_summary.json', DATASET)
    print(json.dumps(export_summary, ensure_ascii=False, indent=2))
else:
    print('Экспорт отключён')

/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/lib/python3.11/site-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset9.py:2855: UserWarning: ONNX export mode is set to TrainingMode.EVAL, but operator 'instance_norm' is set to train=True. Exporting with train=True.
  symbolic_helper.check_training_mode(use_input_stats, "instance_norm")


{
  "checkpoint": "/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_04_training_strategy/weights/final_seeds/control_seed_20260915/final.pt",
  "checkpoint_epoch": 1,
  "config": {
    "epochs": 1,
    "identities_per_batch": 16,
    "images_per_identity": 2,
    "encoder_lr": 0.00011940564013110387,
    "head_lr_multiplier": 10.0,
    "weight_decay": 6.069870050850335e-05,
    "warmup_epochs": 2,
    "min_lr_ratio": 0.02,
    "metric_loss": "supcon",
    "metric_weight": 1.4948762510958067,
    "triplet_margin": 0.5,
    "supcon_temperature": 0.1,
    "consistency_weight": 0.22571745222502657,
    "label_smoothing": 0.1,
    "use_bnneck": true,
    "prefer_cross_camera": true,
    "num_workers": 0,
    "seed": 20260915,
    "pooling": "avg",
    "resize_mode": "square",
    "use_mixstyle": false,
    "mixstyle_probability": 0.5,
    "mixstyle_alpha": 0.1,
    "hard_negative_sampling": false,
    "loss_weight_schedule": "c